%md
# UC-06: Visualisasi Output Crushing Plant (Direct & Non-Direct)

**Sumber Data:** Conveyor BLC + Uscavis

**Karakteristik:** Near real-time (target <= 5 menit)

**Konsumen:** Manajer Crushing Plant, Manajer Quality, Tim Shipping

Mengukur output setiap Crushing Plant berdasarkan data Conveyor BLC dan Uscavis,
dipisahkan menjadi:
- **(a) Direct:** batu bara dari mesin CP langsung dikirim ke conveyor menuju shipment
- **(b) Non-Direct:** batu bara dari CP ditampung terlebih dahulu, lalu dari tampungan diangkut ke conveyor menuju shipment

Konversi volume conveyor -> tonase menggunakan **density BIB = 0.88**

### Arsitektur Data:
```mermaid
flowchart LR
  A[uc.uscavis_raw.webhook_cp_tonages_logs_08] --> I[Silver: uc.uscavis.silver_tonnage_cleaned]
  C[uc.uscavis_raw.webhook_cp_tonages_08]      --> I
  E[uc.uscavis_raw.webhook_cp_items_logs_08]   --> J[Silver: uc.uscavis.silver_cp_status]
  G[uc.synova_raw.cp_queue_assignments...]      --> H[Queue context]
  W[uc.wim.closing_transaction_25_july_2_agustus] --> N[Gold: uc.uscavis.gold_wim_cv_ratio]
  I --> K[Silver: uc.uscavis.silver_output_per_shift]
  I --> L[Gold: uc.uscavis.gold_output_summary]
  I --> N
  J --> L
  K --> L
  I --> M[Gold: uc.uscavis.gold_density_trend]
```


%md
## 1. Imports & Configuration


In [0]:
# ============================================================
# CONFIGURATION - UC06 Crushing Plant Output Visualization
# ============================================================

# Density reference (BIB) - volume to tonnage conversion factor
DENSITY_REFERENCE = 0.88

# Shift definitions (2-shift mining operation)
# Morning: 06:00 AM - 05:59 PM (06:00-17:59)
# Night:   06:00 PM - 05:59 AM (18:00-05:59, crosses midnight)
SHIFT_CONFIG = {
    "Pagi":  {"start": 6, "end": 18, "label": "Morning"},
    "Malam": {"start": 18, "end": 6, "label": "Night"},
}

# Direct vs Non-Direct conveyor classification
# Direct: Conveyor belts that carry coal directly from CP to shipment
# Non-Direct: Conveyor belts that carry coal from stockpile/reclaim area
DIRECT_CONVEYORS = ["CV14", "CV22", "CV12", "CV105", "CV106", "CV107", "CV116"]
NON_DIRECT_CONVEYORS = ["CV15A", "CV15B", "CV23A", "CV23B"]

# CP to conveyor mapping (assumed routing per BIB site layout)
CP_TO_CONVEYOR_DIRECT = {
    "CP1": "CV14",
    "CP6": "CV22",
    "CP7": "CV12",
    "CP8": "CV105",
    "CP9": "CV106",
}

# CP equipment types (for running status)
CP_CHAIN_FEEDERS = [f"CP{i}_CHAIN_FEEDER_STATUS_EQ" for i in [1, 9]]
CP_CHAIN_FEEDERS += [f"CP{i}_CHAIN_FEEDER_STATUS_CONTROL" for i in [3]]
CP_CHAIN_FEEDERS += ["CP2A_CHAIN-FEEDER.FEEDER_ANIMATION", "CP4_Chain_Feeder",
                      "CHAIN_FEEDER_STATUS", "CHAIN_FEEDER_STATUS_EQ",
                      "CP9_CHAIN_FEEDER_STATUS_EQ"]

CP_FEEDER_BREAKERS = [f"CP{i}_FEEDER_BREAKER_STATUS_EQ" for i in [1, 9]]
CP_FEEDER_BREAKERS += [f"CP{i}_FEEDER_BREAKER_STATUS_CONTROL" for i in [3]]
CP_FEEDER_BREAKERS += ["CP2A_FEEDER-BREAKER.BREAKER_ANIMATION", "CP4_Feeder_Breaker",
                        "FEEDER_BREAKER_STATUS", "FEEDER_BREAKER_STATUS_EQ",
                        "CP9_FEEDER_BREAKER_STATUS_EQ"]

# ============================================================
# UNITY CATALOG TABLE REFERENCES
# ============================================================

# Source (Bronze) tables - already ingested in Unity Catalog
SRC_TONNAGE_LOGS = "uc.uscavis_raw.webhook_cp_tonages_logs_08"
SRC_TONNAGE_SNAPSHOT = "uc.uscavis_raw.webhook_cp_tonages_08"
SRC_CP_ITEMS_LOGS = "uc.uscavis_raw.webhook_cp_items_logs_08"
SRC_QUEUE_ASSIGNMENTS = "uc.synova_raw.cp_queue_assignments_358_723_rows_20260803_165205"
SRC_WIM_TRANSACTIONS = "uc.wim.closing_transaction_25_july_2_agustus"

# Output catalog & schema (Silver + Gold)
OUTPUT_CATALOG = "uc"
OUTPUT_SCHEMA = "uscavis"

# Silver table names (written to uc.uscavis)
SILVER_TONNAGE_TABLE = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.silver_tonnage_cleaned"
SILVER_CP_STATUS_TABLE = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.silver_cp_status"
SILVER_OUTPUT_PER_SHIFT_TABLE = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.silver_output_per_shift"

# Gold table names (written to uc.uscavis)
GOLD_OUTPUT_SUMMARY_TABLE = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.gold_output_summary"
GOLD_DENSITY_TREND_TABLE = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.gold_density_trend"
GOLD_WIM_CV_RATIO_TABLE = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.gold_wim_cv_ratio"

print("=" * 60)
print("UC06 - VISUALISASI OUTPUT CRUSHING PLANT")
print("=" * 60)
print(f"  Density Reference (BIB): {DENSITY_REFERENCE}")
print(f"  Direct Conveyors:        {DIRECT_CONVEYORS}")
print(f"  Non-Direct Conveyors:    {NON_DIRECT_CONVEYORS}")
print(f"  CP-Conveyor Mapping:     {CP_TO_CONVEYOR_DIRECT}")
print(f"\n  Source Tables:")
print(f"    Tonnage Logs:    {SRC_TONNAGE_LOGS}")
print(f"    Tonnage Snap:    {SRC_TONNAGE_SNAPSHOT}")
print(f"    CP Items Logs:   {SRC_CP_ITEMS_LOGS}")
print(f"    Queue Assign:    {SRC_QUEUE_ASSIGNMENTS}")
print(f"    WIM Trx:         {SRC_WIM_TRANSACTIONS}")
print(f"\n  Output Schema:     {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}")


In [0]:
import sys
import warnings
from datetime import datetime, timedelta
from functools import reduce
from typing import List, Optional, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, LongType, IntegerType, BooleanType,
)
from pyspark.sql.utils import AnalysisException

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print(f"PySpark version: {pyspark.__version__}")
print(f"Ready for UC06 processing.")


%md
---
## 2. BRONZE LAYER -- Raw Data Ingestion

Load raw data from source files, infer schema, apply minimal transformation,
and persist as Delta tables in the Bronze zone.


%md
### 2.1 Load `webhook_cp_tonages_logs_08.csv` -- Time-Series Conveyor Tonnage


In [0]:
df_tonnage_logs_raw = (
    spark.table(SRC_TONNAGE_LOGS)
    .withColumn("_load_timestamp", F.current_timestamp())
)

n_rows = df_tonnage_logs_raw.count()
n_locations = df_tonnage_logs_raw.select("location").distinct().count()
print(f"{SRC_TONNAGE_LOGS}:")
print(f"  Rows:     {n_rows:,}")
print(f"  Locations: {n_locations}")
df_tonnage_logs_raw.printSchema()
display(df_tonnage_logs_raw.limit(5))


%md
### 2.2 Load `webhook_cp_tonages_08.csv` -- Snapshot Tonnage


In [0]:
df_tonnage_snapshot_raw = (
    spark.table(SRC_TONNAGE_SNAPSHOT)
    .withColumn("_load_timestamp", F.current_timestamp())
)

print(f"{SRC_TONNAGE_SNAPSHOT}: {df_tonnage_snapshot_raw.count():,} rows")
display(df_tonnage_snapshot_raw)


%md
### 2.3 Load `webhook_cp_items_logs_08.csv` -- CP Equipment Status Logs


In [0]:
df_cp_items_raw = (
    spark.table(SRC_CP_ITEMS_LOGS)
    .withColumn("_load_timestamp", F.current_timestamp())
)

n_items = df_cp_items_raw.count()
n_cp = df_cp_items_raw.select("cp_id").distinct().count()
print(f"{SRC_CP_ITEMS_LOGS}:")
print(f"  Rows:    {n_items:,}")
print(f"  CP IDs:  {n_cp}")
display(df_cp_items_raw.limit(10))


%md
### 2.4 Load `cp_queue_assignments.xlsx` -- Truck Queue Assignments


### 2.5 Load WIM Closing Transactions -- Truck Weighbridge Data

WIM (Weigh-in-Motion) data from the truck scale at KM-13.
`netto` column is net coal weight per truck trip in **KG**.

In [0]:
df_wim_raw = (
    spark.table(SRC_WIM_TRANSACTIONS)
    .select(
        "id", "truck", "driver", "material", "coal",
        "netto", "gross", "tare",
        "timestamp_gross_local", "timestamp_in_local", "timestamp_out_local",
        "location_in", "location_closing", "data_source",
    )
    .withColumn("_load_timestamp", F.current_timestamp())
)

n_wim = df_wim_raw.count()
print(f"{SRC_WIM_TRANSACTIONS}:")
print(f"  Rows:      {n_wim:,}")
print(f"  Locations: {df_wim_raw.select('location_in').distinct().collect()}")
print(f"  Date Range: {df_wim_raw.select(F.min('timestamp_gross_local'), F.max('timestamp_gross_local')).first()}")
display(df_wim_raw.limit(10))

In [0]:
df_queue_raw = (
    spark.table(SRC_QUEUE_ASSIGNMENTS)
    .withColumn("_load_timestamp", F.current_timestamp())
)

print(f"{SRC_QUEUE_ASSIGNMENTS}: {df_queue_raw.count():,} rows")
display(df_queue_raw.limit(10))


%md
### 2.5 Write Bronze Delta Tables


In [0]:
# Bronze layer validation -- data is already in Unity Catalog
# No need to write Bronze tables; source tables serve as the Bronze layer.
print("=" * 60)
print("BRONZE LAYER VALIDATION (Source UC Tables)")
print("=" * 60)

bronze_tables = {
    "tonnage_logs": (SRC_TONNAGE_LOGS, df_tonnage_logs_raw),
    "tonnage_snapshot": (SRC_TONNAGE_SNAPSHOT, df_tonnage_snapshot_raw),
    "cp_items_logs": (SRC_CP_ITEMS_LOGS, df_cp_items_raw),
    "queue_assignments": (SRC_QUEUE_ASSIGNMENTS, df_queue_raw),
    "wim_transactions": (SRC_WIM_TRANSACTIONS, df_wim_raw),
}

for name, (table_ref, df) in bronze_tables.items():
    count = df.count()
    print(f"  [OK] {name:<20} -> {table_ref} ({count:,} rows)")

print("\nBronze layer validation complete. Source data ready for Silver transforms.")


%md
---
## 3. SILVER LAYER -- Data Cleansing & Transformation

Clean raw data, parse timestamps, deduplicate, compute incremental tonnage,
classify direct vs non-direct flow, convert volume to tonnage, and assign shifts.


%md
### 3.1 Data Profiling & Quality Audit


In [0]:
def profile_dataframe(df, name):
    """Audit dataframe: nulls, duplicates, schema, value ranges."""
    print(f"\n{'='*60}")
    print(f"PROFILING: {name}")
    print(f"{'='*60}")

    n = df.count()
    print(f"Total rows: {n:,}")
    print(f"Total cols: {len(df.columns)}")

    print("\nNullable columns:")
    for col_name in df.columns:
        null_cnt = df.filter(F.col(col_name).isNull()).count()
        if null_cnt > 0:
            pct = null_cnt / n * 100
            print(f"  {col_name}: {null_cnt:,} null ({pct:.2f}%)")

    print("\nDuplicate rows (all columns):")
    dup_cnt = df.groupBy(df.columns).count().filter(F.col("count") > 1).count()
    print(f"  Duplicates: {dup_cnt:,}")

    df.show(5, truncate=False)

profile_dataframe(df_tonnage_logs_raw, "webhook_cp_tonages_logs")
profile_dataframe(df_cp_items_raw, "webhook_cp_items_logs")


%md
### 3.2 Clean & Transform Tonnage Logs


In [0]:
# Step 1: Rename timestamp columns (already TIMESTAMP type from UC table)
# Step 2: Cast numeric columns to DoubleType
# Step 3: Classify location type (CP vs CV)
# Step 4: Deduplicate per location per timestamp (keep latest created_at)

df_tonnage_cleaned = df_tonnage_logs_raw \
    .dropDuplicates() \
    .withColumn("timestamp_data_ts", F.col("timestamp_data").cast("timestamp")) \
    .withColumn("created_at_ts", F.col("created_at").cast("timestamp")) \
    .withColumn("totalizer_this_month", F.col("totalizer_this_month").cast(DoubleType())) \
    .withColumn("totalizer_last_month", F.col("totalizer_last_month").cast(DoubleType())) \
    .withColumn("totalizer_today", F.col("totalizer_today").cast(DoubleType())) \
    .withColumn("totalizer_total_month", F.col("totalizer_total_month").cast(DoubleType())) \
    .withColumn("location_type",
        F.when(F.col("location").rlike("^CP"), F.lit("Crushing Plant"))
         .when(F.col("location").rlike("^CV"), F.lit("Conveyor"))
         .otherwise(F.col("location"))
    ) \
    .withColumn("is_cp", F.col("location").rlike("^CP")) \
    .withColumn("is_cv", F.col("location").rlike("^CV")) \
    .withColumn("location_group",
        F.when(F.col("is_cp"), F.col("location"))
         .when(
             F.col("location").isin(NON_DIRECT_CONVEYORS),
             F.lit("NON_DIRECT_RECLAIM")
         )
         .otherwise(F.lit("DIRECT_SHIPMENT"))
    ) \
    .drop("_load_timestamp")

# Deduplicate: keep only the latest record per (location, timestamp)
window_dedup = Window.partitionBy("location", "timestamp_data_ts") \
    .orderBy(F.col("created_at_ts").desc())

df_tonnage_cleaned = df_tonnage_cleaned \
    .withColumn("_rn", F.row_number().over(window_dedup)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn")

print(f"Raw tonnage rows:        {df_tonnage_logs_raw.count():,}")
print(f"Cleaned tonnage rows:    {df_tonnage_cleaned.count():,}")
display(df_tonnage_cleaned.limit(10))


%md
### 3.3 Compute Incremental Tonnage (Delta from cumulative totalizer)

The `totalizer_today` value is a **running cumulative counter**. We compute the
difference between consecutive readings to determine the incremental volume per interval.


In [0]:
def compute_delta_tonnage(df, location_col="location",
                          ts_col="timestamp_data_ts",
                          totalizer_col="totalizer_today",
                          delta_vol_col="delta_volume",
                          delta_time_col="time_delta_seconds"):
    """
    Compute incremental volume (delta) from cumulative totalizer readings.

    Handles edge cases:
    - Null current/previous values
    - Totalizer reset/rollover (new day starts with lower value)
    - Negative deltas (filtered out)
    """
    window_spec = Window.partitionBy(location_col).orderBy(ts_col)

    df_result = df \
        .withColumn("_prev_total", F.lag(totalizer_col).over(window_spec)) \
        .withColumn("_prev_ts", F.lag(ts_col).over(window_spec)) \
        .withColumn(
            delta_time_col,
            F.col(ts_col).cast("long") - F.col("_prev_ts").cast("long")
        ) \
        .withColumn(
            delta_vol_col,
            F.when(
                F.col(totalizer_col).isNull() | F.col("_prev_total").isNull(),
                F.lit(None)
            ).when(
                F.col(totalizer_col) < F.col("_prev_total"),
                F.lit(None)  # Rollover -- cannot compute meaningful delta
            ).otherwise(
                (F.col(totalizer_col) - F.col("_prev_total")).cast(DoubleType())
            )
        ) \
        .withColumn(
            delta_vol_col,
            F.when(F.col(delta_vol_col).isNull(), F.lit(None))
             .when(F.col(delta_vol_col) < 0, F.lit(None))
             .otherwise(F.col(delta_vol_col))
        ) \
        .drop("_prev_total", "_prev_ts")

    return df_result

df_tonnage_with_delta = compute_delta_tonnage(df_tonnage_cleaned)

valid_deltas = df_tonnage_with_delta.filter(
    F.col("delta_volume").isNotNull()
).count()
print(f"Records with valid delta: {valid_deltas:,} "
      f"({valid_deltas / df_tonnage_with_delta.count() * 100:.1f}%)")

display(df_tonnage_with_delta.select(
    "location", "timestamp_data_ts", "totalizer_today",
    "delta_volume", "time_delta_seconds"
).limit(20))


%md
### 3.4 Classify: Direct vs Non-Direct Flow Mode


In [0]:
# Classify each reading by flow mode:
#   'direct'      - Conveyor belts directly connected to CP -> shipment
#   'non_direct'  - Conveyor belts that reclaim from stockpile
#   'cp_output'   - CP totalizer reading (represents CP production)

df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn(
        "flow_mode",
        F.when(F.col("location").isin(DIRECT_CONVEYORS), F.lit("direct"))
         .when(F.col("location").isin(NON_DIRECT_CONVEYORS), F.lit("non_direct"))
         .when(F.col("is_cp"), F.lit("cp_output"))
         .otherwise(F.lit("other"))
    )

# Map conveyors back to their source CP (for traceability)
df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn("linked_cp", F.lit(None).cast(StringType()))

for cp_name, cv_name in CP_TO_CONVEYOR_DIRECT.items():
    df_tonnage_with_delta = df_tonnage_with_delta \
        .withColumn(
            "linked_cp",
            F.when(F.col("location") == cv_name, F.lit(cp_name))
             .otherwise(F.col("linked_cp"))
        )

# For CP locations, linked_cp = their own name
df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn(
        "linked_cp",
        F.when(F.col("is_cp"), F.col("location"))
         .otherwise(F.col("linked_cp"))
    )

display(df_tonnage_with_delta.groupBy("location", "flow_mode", "location_type")
        .count()
        .orderBy("location"))


%md
### 3.5 Volume -> Tonnage Conversion (x density BIB 0.88)

The totalizer measures **volume** (from belt load cell / BLC). Tonnage is derived as:

$$ \text{Tonnage} = \text{Volume} \times \text{Density}_{\text{BIB}} $$


In [0]:
df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn("tonnage_delta", F.round(F.col("delta_volume") * DENSITY_REFERENCE, 4)) \
    .withColumn("tonnage_total_today", F.round(F.col("totalizer_today") * DENSITY_REFERENCE, 2)) \
    .withColumn("tonnage_total_this_month", F.round(F.col("totalizer_this_month") * DENSITY_REFERENCE, 2)) \
    .withColumn("tonnage_total_last_month", F.round(F.col("totalizer_last_month") * DENSITY_REFERENCE, 2)) \
    .withColumn("tonnage_total_all", F.round(F.col("totalizer_total_month") * DENSITY_REFERENCE, 2))

# Compute instantaneous throughput rate (ton/hour)
df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn(
        "throughput_tph",
        F.when(
            (F.col("time_delta_seconds").isNotNull()) & (F.col("time_delta_seconds") > 0),
            F.round(F.col("tonnage_delta") / (F.col("time_delta_seconds") / 3600.0), 2)
        ).otherwise(F.lit(None))
    )

print(f"Volume -> Tonnage conversion applied (x {DENSITY_REFERENCE})")

display(df_tonnage_with_delta.select(
    "location", "timestamp_data_ts", "delta_volume",
    "tonnage_delta", "throughput_tph", "tonnage_total_today"
).filter(F.col("tonnage_delta").isNotNull()).limit(10))


%md
### 3.6 Shift Assignment


In [0]:
def assign_shift_col(ts_col):
    """
    Assign shift based on hour.
    Pagi  (Morning): 06:00 AM - 05:59 PM  (06:00-17:59)
    Malam (Night):   06:00 PM - 05:59 AM  (18:00-05:59, crosses midnight)
    """
    hour = F.hour(ts_col)
    return (
        F.when((hour >= 6) & (hour < 18), F.lit("Pagi"))
         .otherwise(F.lit("Malam"))
    )

df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn("shift", assign_shift_col(F.col("timestamp_data_ts"))) \
    .withColumn("event_date", F.to_date(F.col("timestamp_data_ts")))

# Malam (Night) shift crosses midnight -> adjust to previous calendar date for grouping
df_tonnage_with_delta = df_tonnage_with_delta \
    .withColumn(
        "adjusted_date",
        F.when(
            (F.col("shift") == "Malam") & (F.hour(F.col("timestamp_data_ts")) < 6),
            F.date_sub(F.col("event_date"), 1)
        ).otherwise(F.col("event_date"))
    )

display(df_tonnage_with_delta.groupBy("shift", "flow_mode").count()
        .orderBy("shift", "flow_mode"))


%md
### 3.7 CP Equipment Status -> Running Hours


In [0]:
# Map cp_id -> CP Name
cp_id_map = {
    1: "CP1", 2: "CP2", 3: "CP3", 4: "CP4", 5: "CP5",
    6: "CP6", 7: "CP7", 8: "CP8", 9: "CP9", 11: "CP2New",
}

_cond = F.lit(None).cast(StringType())
for cid, cname in cp_id_map.items():
    _cond = F.when(F.col("cp_id") == cid, F.lit(cname)).otherwise(_cond)
_cond = F.when(_cond.isNull(), F.concat(F.lit("CP"), F.col("cp_id").cast("string"))).otherwise(_cond)

df_cp_status_clean = df_cp_items_raw \
    .dropDuplicates() \
    .withColumn("created_at_ts", F.to_timestamp(F.col("created_at"))) \
    .withColumn("cp_number", _cond) \
    .withColumn("equipment_category",
        F.when(F.col("name").isin(CP_CHAIN_FEEDERS), F.lit("chain_feeder"))
         .when(F.col("name").isin(CP_FEEDER_BREAKERS), F.lit("feeder_breaker"))
         .when(F.col("name").rlike("(?i)crusher"), F.lit("crusher"))
         .when(F.col("name").rlike("(?i)conveyor"), F.lit("conveyor"))
         .otherwise(F.lit("other"))
    ) \
    .withColumn("is_running", F.when(F.col("status") == "Running", True).otherwise(False)) \
    .withColumn("shift", assign_shift_col(F.col("created_at_ts"))) \
    .withColumn("event_date", F.to_date(F.col("created_at_ts"))) \
    .drop("_load_timestamp")

# Compute running hours per CP per equipment per shift
# Assumption: ~30 second interval between consecutive readings
READING_INTERVAL_SEC = 30.0

df_cp_running_stats = df_cp_status_clean.groupBy(
    "cp_number", "event_date", "shift", "equipment_category"
).agg(
    F.count(F.lit(1)).alias("total_readings"),
    F.sum(F.col("is_running").cast("int")).alias("running_readings"),
).withColumn(
    "running_hours_est",
    F.round((F.col("running_readings") * READING_INTERVAL_SEC) / 3600.0, 2)
).withColumn(
    "running_pct",
    F.round(F.col("running_readings") / F.col("total_readings") * 100, 2)
).withColumn(
    "utilization",
    F.round(F.col("running_hours_est") / 12.0 * 100, 2)  # 12-hour shift
)

display(df_cp_running_stats.orderBy("event_date", "cp_number", "shift").limit(20))


%md
### 3.8 Aggregate Output per CP per Shift per Day


In [0]:
# Aggregate incremental tonnage by CP location, shift, and adjusted date
df_output_per_shift = df_tonnage_with_delta \
    .filter(F.col("is_cp") & F.col("tonnage_delta").isNotNull()) \
    .groupBy("location", "adjusted_date", "shift") \
    .agg(
        F.count(F.lit(1)).alias("reading_count"),
        F.round(F.sum("delta_volume"), 2).alias("total_volume"),
        F.round(F.sum("tonnage_delta"), 2).alias("total_tonnage"),
        F.round(F.avg("delta_volume"), 2).alias("avg_volume_per_reading"),
        F.round(F.avg("tonnage_delta"), 2).alias("avg_tonnage_per_reading"),
        F.round(F.max("tonnage_total_today"), 2).alias("totalizer_tonnage_eod"),
        F.round(F.avg("throughput_tph"), 2).alias("avg_throughput_tph"),
        F.first("timestamp_data_ts").alias("first_reading"),
        F.last("timestamp_data_ts").alias("last_reading"),
    ) \
    .withColumn("duration_hours",
        F.round(
            (F.col("last_reading").cast("long") - F.col("first_reading").cast("long")) / 3600.0,
            2
        )
    ) \
    .withColumnRenamed("location", "cp_name")

print(f"Output per shift rows: {df_output_per_shift.count():,}")
display(df_output_per_shift.orderBy("adjusted_date", "cp_name", "shift").limit(20))


%md
### 3.9 Aggregate per Conveyor (Direct vs Non-Direct)


In [0]:
# Aggregate conveyor data by flow mode
df_conveyor_per_shift = df_tonnage_with_delta \
    .filter(F.col("is_cv") & F.col("tonnage_delta").isNotNull()) \
    .groupBy("location", "flow_mode", "adjusted_date", "shift") \
    .agg(
        F.count(F.lit(1)).alias("reading_count"),
        F.round(F.sum("tonnage_delta"), 2).alias("total_tonnage"),
        F.round(F.avg("throughput_tph"), 2).alias("avg_throughput_tph"),
        F.round(F.max("tonnage_total_today"), 2).alias("totalizer_tonnage_eod"),
    )

display(df_conveyor_per_shift.orderBy("adjusted_date", "shift", "location").limit(20))


%md
### 3.10 Write Silver Delta Tables


In [0]:
def write_silver(df, table_full_name):
    """Write dataframe to Silver Delta table in Unity Catalog."""
    print(f"Writing {table_full_name} ...")
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_full_name)
    count = df.count()
    print(f"  [OK] {table_full_name} -- {count:,} rows written.")

write_silver(df_tonnage_with_delta, SILVER_TONNAGE_TABLE)
write_silver(df_cp_status_clean, SILVER_CP_STATUS_TABLE)
write_silver(df_output_per_shift, SILVER_OUTPUT_PER_SHIFT_TABLE)

print("\nSilver layer transformation complete.")


%md
---
## 4. ANALYTICS & VISUALIZATIONS

=== KEY METRICS (UC-06) ===
1. **Tonase output per CP per shift** (direct vs non-direct)
2. **Pie chart: proporsi direct vs non-direct**
3. **Konsistensi reading: rasio tonase WIM vs tonase conveyor** (deteksi loss/anomali)
4. **Trend density realisasi vs density referensi (0.88)** -- indikator kualitas


%md
### 4.1 Tonase Output per CP per Shift -- Bar Chart


In [0]:
pdf_cp_shift = df_output_per_shift \
    .filter(F.col("total_tonnage") > 0) \
    .orderBy("adjusted_date", "shift", "cp_name") \
    .toPandas()

if not pdf_cp_shift.empty:
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # --- (A) Total tonnage per CP ---
    cp_agg = pdf_cp_shift.groupby("cp_name")["total_tonnage"].sum().sort_values(ascending=False)
    bars = axes[0].bar(cp_agg.index, cp_agg.values, color="steelblue", edgecolor="black")
    axes[0].set_title("Total Tonase Output per Crushing Plant")
    axes[0].set_xlabel("Crushing Plant")
    axes[0].set_ylabel("Tonase (ton)")
    axes[0].tick_params(axis="x", rotation=45)
    for bar, val in zip(bars, cp_agg.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + cp_agg.max() * 0.01,
                     f"{val:,.0f}", ha="center", fontsize=8)

    # --- (B) Per shift per CP ---
    shift_pivot = pdf_cp_shift.pivot_table(
        index="cp_name", columns="shift", values="total_tonnage", aggfunc="sum"
    ).fillna(0)
    shift_colors = {"Pagi": "#2196F3", "Malam": "#FF9800"}
    shift_pivot.plot(
        kind="bar", ax=axes[1], edgecolor="black",
        color=[shift_colors.get(s, "#888") for s in shift_pivot.columns]
    )
    axes[1].set_title("Tonase Output per CP per Shift")
    axes[1].set_xlabel("Crushing Plant")
    axes[1].set_ylabel("Tonase (ton)")
    axes[1].tick_params(axis="x", rotation=45)

    # --- (C) Daily trend per CP ---
    daily_pivot = pdf_cp_shift.pivot_table(
        index="adjusted_date", columns="cp_name", values="total_tonnage", aggfunc="sum"
    ).fillna(0)
    daily_pivot.plot(ax=axes[2], marker="o", markersize=4)
    axes[2].set_title("Daily Output Trend per CP")
    axes[2].set_xlabel("Date")
    axes[2].set_ylabel("Tonase (ton)")
    axes[2].legend(title="CP", fontsize=8)
    axes[2].tick_params(axis="x", rotation=45)
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))

    plt.tight_layout()
    plt.show()
else:
    print("[INFO] No CP output data available for plotting.")


%md
---
### 4.2 Pie Chart: Proporsi Direct vs Non-Direct


In [0]:
pdf_flow = df_tonnage_with_delta \
    .filter(F.col("tonnage_delta").isNotNull()) \
    .groupBy("flow_mode") \
    .agg(
        F.round(F.sum("tonnage_delta"), 2).alias("total_tonnage"),
        F.count(F.lit(1)).alias("reading_count"),
    ) \
    .filter(F.col("total_tonnage") > 0) \
    .toPandas()

if not pdf_flow.empty:
    flow_labels = {
        "cp_output": "CP Output\n(Totalizer CP)",
        "direct": "Direct Flow\n(Conveyor -> Shipment)",
        "non_direct": "Non-Direct Flow\n(Stockpile -> Reclaim -> Shipment)",
        "other": "Other",
    }
    pdf_flow["label"] = pdf_flow["flow_mode"].map(flow_labels)
    colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]
    explode = [0.03] * len(pdf_flow)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

    # Pie chart
    wedges, texts, autotexts = ax1.pie(
        pdf_flow["total_tonnage"], labels=pdf_flow["label"],
        autopct="%1.1f%%", startangle=140,
        colors=colors[:len(pdf_flow)],
        explode=explode,
        textprops={"fontsize": 9},
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_fontweight("bold")
    ax1.set_title("Proporsi Output: Direct vs Non-Direct Flow")

    # Bar chart
    bars = ax2.barh(pdf_flow["label"], pdf_flow["total_tonnage"],
                   color=colors[:len(pdf_flow)], edgecolor="black")
    ax2.set_title("Total Tonase per Flow Mode")
    ax2.set_xlabel("Tonase (ton)")
    for bar, val in zip(bars, pdf_flow["total_tonnage"]):
        ax2.text(bar.get_width() + pdf_flow["total_tonnage"].max() * 0.01,
                bar.get_y() + bar.get_height()/2,
                f"{val:,.0f} ton", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

    # Calculate proportion
    total_all = pdf_flow["total_tonnage"].sum()
    print(f"\n{'='*40}")
    print(f"FLOW MODE DISTRIBUTION")
    print(f"{'='*40}")
    for _, row in pdf_flow.iterrows():
        pct = row["total_tonnage"] / total_all * 100
        print(f"  {row['label']:40s}: {row['total_tonnage']:>12,.0f} ton ({pct:5.1f}%)")
    print(f"  {'TOTAL':40s}: {total_all:>12,.0f} ton (100.0%)")
else:
    print("[INFO] No flow mode data available.")


%md
---
### 4.3 Konsistensi Reading: Rasio Tonase WIM vs Conveyor

**Purpose:** Detect loss/anomali dengan membandingkan tonase terukur dari
**WIM (Weigh-in-Motion)** di truck scale (KM-13) vs tonase dari **Conveyor BLC**.

$ \text{Rasio WIM/CV} = \frac{\Sigma \text{Tonase WIM}_{\text{truck (netto)}}}{\Sigma \text{Tonase Conveyor}_{\text{BLC}}} $

- Rasio ideal ≈ 1.0 (kedua sistem sinkron)
- Rasio < 1.0: ada loss material di conveyor (spillage, penguapan)
- Rasio > 1.0: kemungkinan double-counting atau kalibrasi WIM tidak tepat
- Deviasi > ±5% dari 1.0 perlu investigasi

**Data Source:** `uc.wim.closing_transaction_25_july_2_agustus` — netto weight in KG per truck trip.


In [0]:
# ----------------------------------------------------------------
# WIM vs Conveyor Ratio -- Actual Computation
# ----------------------------------------------------------------

# Step 1: Aggregate WIM tonnage per date (netto is in KG -> convert to Ton)
df_wim_daily = df_wim_raw \
    .withColumn("wim_date", F.to_date("timestamp_gross_local")) \
    .filter(F.col("netto").isNotNull() & (F.col("netto") > 0)) \
    .groupBy("wim_date") \
    .agg(
        F.round(F.sum(F.col("netto") / 1000.0), 2).alias("wim_tonnage"),
        F.count("id").alias("truck_count"),
        F.round(F.avg(F.col("netto") / 1000.0), 2).alias("avg_netto_per_truck"),
    )

print("WIM Daily Tonnage (from truck scale at KM-13):")
display(df_wim_daily.orderBy("wim_date"))

# Step 2: Aggregate conveyor tonnage per date (all CVs combined)
df_conveyor_daily = df_tonnage_with_delta \
    .filter(F.col("is_cv") & F.col("tonnage_delta").isNotNull()) \
    .groupBy("adjusted_date") \
    .agg(
        F.round(F.sum("tonnage_delta"), 2).alias("conveyor_tonnage"),
        F.round(F.sum("delta_volume"), 2).alias("conveyor_volume"),
        F.count(F.lit(1)).alias("cv_reading_count"),
    )

print("\nConveyor Daily Tonnage (from BLC totalizer):")
display(df_conveyor_daily.orderBy("adjusted_date"))


In [0]:
# ----------------------------------------------------------------
# Step 3: Join WIM daily with Conveyor daily and compute ratio
# ----------------------------------------------------------------

df_wim_cv_ratio = df_wim_daily.alias("wim") \
    .join(
        df_conveyor_daily.alias("cv"),
        F.col("wim.wim_date") == F.col("cv.adjusted_date"),
        "inner"
    ) \
    .select(
        F.col("wim.wim_date").alias("date"),
        F.col("wim.wim_tonnage"),
        F.col("wim.truck_count"),
        F.col("wim.avg_netto_per_truck"),
        F.col("cv.conveyor_tonnage"),
        F.col("cv.conveyor_volume"),
        F.col("cv.cv_reading_count"),
    ) \
    .withColumn(
        "wim_cv_ratio",
        F.round(F.col("wim_tonnage") / F.col("conveyor_tonnage"), 4)
    ) \
    .withColumn(
        "deviation_pct",
        F.round((F.col("wim_cv_ratio") - 1.0) * 100, 2)
    ) \
    .withColumn(
        "anomaly_flag",
        F.when(F.abs(F.col("wim_cv_ratio") - 1.0) > 0.05, F.lit(True))
         .otherwise(F.lit(False))
    ) \
    .withColumn(
        "anomaly_type",
        F.when(F.col("wim_cv_ratio") > 1.05, F.lit("WIM_OVER"))
         .when(F.col("wim_cv_ratio") < 0.95, F.lit("CV_OVER"))
         .otherwise(F.lit("NORMAL"))
    ) \
    .withColumn(
        "density_actual",
        F.when(F.col("conveyor_volume") > 0,
               F.round(F.col("wim_tonnage") / F.col("conveyor_volume"), 4))
         .otherwise(F.lit(None))
    )

print("WIM vs Conveyor Ratio (daily):")
print("  Anomaly threshold: |ratio - 1.0| > 5%")
display(df_wim_cv_ratio.orderBy("date"))

# Summary stats
anomalies = df_wim_cv_ratio.filter(F.col("anomaly_flag") == True).count()
total_days = df_wim_cv_ratio.count()
print(f"\nResult: {anomalies} anomaly day(s) out of {total_days} overlapping days.")


In [0]:
# ----------------------------------------------------------------
# WIM vs Conveyor Ratio Visualization
# ----------------------------------------------------------------

pdf_ratio = df_wim_cv_ratio.orderBy("date").toPandas()

if not pdf_ratio.empty:
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # --- (A) Daily WIM/CV ratio with tolerance band ---
    axes[0].plot(pdf_ratio["date"], pdf_ratio["wim_cv_ratio"],
                marker="o", linewidth=2, color="#1565C0", label="WIM/CV Ratio")
    axes[0].axhline(y=1.0, color="green", linestyle="-", linewidth=2, label="Ideal = 1.0")
    axes[0].axhspan(0.95, 1.05, alpha=0.15, color="green", label="±5% tolerance")
    axes[0].axhline(y=1.05, color="red", linestyle="--", linewidth=1, alpha=0.7)
    axes[0].axhline(y=0.95, color="red", linestyle="--", linewidth=1, alpha=0.7)
    # Mark anomalies
    anomaly_mask = pdf_ratio["anomaly_flag"] == True
    if anomaly_mask.any():
        axes[0].scatter(pdf_ratio.loc[anomaly_mask, "date"],
                       pdf_ratio.loc[anomaly_mask, "wim_cv_ratio"],
                       color="red", s=100, zorder=5, label="Anomaly")
    axes[0].set_title("Daily WIM/CV Ratio (Konsistensi Reading)")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Ratio (WIM Tonnage / CV Tonnage)")
    axes[0].legend(fontsize=9)
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))

    # --- (B) Bar chart: WIM vs Conveyor tonnage side-by-side ---
    x = range(len(pdf_ratio))
    width = 0.35
    axes[1].bar([i - width/2 for i in x], pdf_ratio["wim_tonnage"],
               width, label="WIM Tonnage", color="#FF8F00", edgecolor="black")
    axes[1].bar([i + width/2 for i in x], pdf_ratio["conveyor_tonnage"],
               width, label="Conveyor Tonnage", color="#1565C0", edgecolor="black")
    axes[1].set_title("Daily Tonnage: WIM vs Conveyor")
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Tonnage (Ton)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([d.strftime("%m/%d") for d in pdf_ratio["date"]], rotation=45)
    axes[1].legend()

    # --- (C) Deviation percentage bar chart ---
    colors = ["#F44336" if abs(v) > 5 else "#4CAF50" for v in pdf_ratio["deviation_pct"]]
    axes[2].bar(range(len(pdf_ratio)), pdf_ratio["deviation_pct"],
               color=colors, edgecolor="black")
    axes[2].axhline(y=0, color="black", linewidth=1)
    axes[2].axhline(y=5, color="red", linestyle="--", alpha=0.7, label="+5% threshold")
    axes[2].axhline(y=-5, color="red", linestyle="--", alpha=0.7, label="-5% threshold")
    axes[2].set_title("Deviasi WIM/CV dari Ideal (%)")
    axes[2].set_xlabel("Date")
    axes[2].set_ylabel("Deviation (%)")
    axes[2].set_xticks(range(len(pdf_ratio)))
    axes[2].set_xticklabels([d.strftime("%m/%d") for d in pdf_ratio["date"]], rotation=45)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\n{'='*60}")
    print(f"WIM vs CONVEYOR RATIO SUMMARY")
    print(f"{'='*60}")
    print(f"  Period:        {pdf_ratio['date'].min()} to {pdf_ratio['date'].max()}")
    print(f"  Days analyzed: {len(pdf_ratio)}")
    print(f"  Avg ratio:     {pdf_ratio['wim_cv_ratio'].mean():.4f}")
    print(f"  Min ratio:     {pdf_ratio['wim_cv_ratio'].min():.4f}")
    print(f"  Max ratio:     {pdf_ratio['wim_cv_ratio'].max():.4f}")
    print(f"  Anomaly days:  {anomaly_mask.sum()} ({anomaly_mask.sum()/len(pdf_ratio)*100:.0f}%)")
    print(f"{'='*60}")
else:
    print("No overlapping WIM + Conveyor data found for ratio analysis.")


%md
---
### 4.4 Trend Density Realisasi vs Density Referensi (0.88)

**Purpose:** Bandingkan density aktual (dihitung dari WIM + conveyor volume)
terhadap density referensi BIB (0.88).

$$ \text{Density}_{\text{aktual}} = \frac{\text{Tonase WIM}_{\text{truck}}}{\text{Volume Conveyor}_{\text{BLC}}} $$

- Density aktual >> 0.88: batubara lebih padat / WIM over-kalibrasi
- Density aktual << 0.88: batubara lebih ringan (kualitas lebih rendah) / volume conveyor over-estimasi
- Trend menurun density -> indikasi penurunan kualitas batubara


In [0]:
# ----------------------------------------------------------------
# Density Trend Visualization
# ----------------------------------------------------------------

# For now, compute realized density using available data:
# realized_density = conveyor_tonnage / conveyor_volume (should approach 0.88)
# The conveyor system reports volume; we validate against the reference density.

pdf_density_trend = df_tonnage_with_delta \
    .filter(
        F.col("is_cv") &
        F.col("tonnage_delta").isNotNull() &
        F.col("delta_volume").isNotNull() &
        (F.col("delta_volume") > 0)
    ) \
    .groupBy("adjusted_date", "location", "flow_mode") \
    .agg(
        F.round(F.sum("tonnage_delta"), 2).alias("total_tonnage"),
        F.round(F.sum("delta_volume"), 2).alias("total_volume"),
    ) \
    .withColumn(
        "realized_density",
        F.when(F.col("total_volume") > 0,
               F.round(F.col("total_tonnage") / F.col("total_volume"), 4))
         .otherwise(F.lit(None))
    ) \
    .withColumn(
        "density_deviation",
        F.round(F.col("realized_density") - DENSITY_REFERENCE, 4)
    ) \
    .withColumn(
        "density_deviation_pct",
        F.round((F.col("realized_density") - DENSITY_REFERENCE) / DENSITY_REFERENCE * 100, 2)
    ) \
    .filter(F.col("realized_density").isNotNull()) \
    .toPandas()

if not pdf_density_trend.empty:
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # --- (A) Density trend per conveyor per date ---
    for loc in pdf_density_trend["location"].unique()[:5]:  # Top 5 conveyors
        subset = pdf_density_trend[pdf_density_trend["location"] == loc].sort_values("adjusted_date")
        axes[0].plot(subset["adjusted_date"], subset["realized_density"],
                    marker="o", markersize=4, label=loc)
    axes[0].axhline(y=DENSITY_REFERENCE, color="red", linestyle="--", linewidth=2,
                    label=f"Referensi BIB = {DENSITY_REFERENCE}")
    axes[0].set_title("Trend Density Realisasi per Conveyor vs Referensi BIB")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Density (ton/m3 or ton/unit volume)")
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))

    # --- (B) Deviation from reference (%) ---
    pdf_density_trend_sorted = pdf_density_trend.sort_values("density_deviation_pct")
    bars = axes[1].barh(
        pdf_density_trend_sorted["location"] + "\n(" + pdf_density_trend_sorted["adjusted_date"].astype(str) + ")",
        pdf_density_trend_sorted["density_deviation_pct"],
        color=["#4CAF50" if v >= 0 else "#F44336" for v in pdf_density_trend_sorted["density_deviation_pct"]],
        edgecolor="black",
    )
    axes[1].axvline(x=0, color="black", linewidth=1.5)
    axes[1].set_title("Deviasi Density dari Referensi (%)")
    axes[1].set_xlabel("Deviation (%)")
    axes[1].tick_params(axis="y", labelsize=7)

    # --- (C) Boxplot: density distribution by conveyor ---
    pivot_data = [pdf_density_trend[pdf_density_trend["location"] == loc]["realized_density"]
                 for loc in pdf_density_trend["location"].unique()
                 if len(pdf_density_trend[pdf_density_trend["location"] == loc]) > 1]
    pivot_labels = [loc for loc in pdf_density_trend["location"].unique()
                   if len(pdf_density_trend[pdf_density_trend["location"] == loc]) > 1]
    if pivot_data:
        axes[2].boxplot(pivot_data, labels=pivot_labels, patch_artist=True)
        axes[2].axhline(y=DENSITY_REFERENCE, color="red", linestyle="--", linewidth=2,
                       label=f"Referensi = {DENSITY_REFERENCE}")
        axes[2].set_title("Distribusi Density Realisasi per Conveyor")
        axes[2].set_ylabel("Density")
        axes[2].tick_params(axis="x", rotation=45, labelsize=8)
        axes[2].legend()

    plt.tight_layout()
    plt.show()

    # Summary stats
    print(f"\n{'='*55}")
    print(f"DENSITY REALISASI SUMMARY")
    print(f"{'='*55}")
    for _, row in pdf_density_trend.iterrows():
        flag = " [ANOMALI]" if abs(row["density_deviation_pct"]) > 5 else ""
        print(
            f"  {row['location']:10s} | {row['adjusted_date']:%Y-%m-%d} | "
            f"density={row['realized_density']:.4f} | "
            f"dev={row['density_deviation_pct']:+.2f}%{flag}"
        )
else:
    print("[INFO] No density trend data available. This metric requires conveyor volume and tonnage data with valid deltas.")


%md
---
## 5. GOLD LAYER -- Summary & Export


%md
### 5.1 Output Summary per CP per Shift (Gold)


In [0]:
# Join CP output with CP running status for comprehensive summary
df_gold_summary = df_output_per_shift.alias("out") \
    .join(
        df_cp_running_stats.groupBy("cp_number", "event_date", "shift")
        .agg(F.round(F.avg("running_hours_est"), 2).alias("avg_running_hours"),
             F.round(F.avg("running_pct"), 2).alias("avg_running_pct"))
        .alias("stat"),
        (F.col("out.cp_name") == F.col("stat.cp_number")) &
        (F.col("out.adjusted_date") == F.col("stat.event_date")) &
        (F.col("out.shift") == F.col("stat.shift")),
        "left"
    ) \
    .select(
        F.col("out.adjusted_date").alias("date"),
        F.col("out.shift").alias("shift"),
        F.col("out.cp_name").alias("cp_name"),
        F.col("out.total_tonnage").alias("total_tonnage_ton"),
        F.col("out.total_volume").alias("total_volume"),
        F.col("out.avg_throughput_tph").alias("avg_throughput_tph"),
        F.col("out.duration_hours").alias("reading_duration_hours"),
        F.coalesce(F.col("stat.avg_running_hours"), F.lit(0)).alias("running_hours"),
        F.coalesce(F.col("stat.avg_running_pct"), F.lit(0)).alias("running_pct"),
    ) \
    .withColumn("productivity_tph",
        F.when(F.col("running_hours") > 0,
               F.round(F.col("total_tonnage_ton") / F.col("running_hours"), 2))
         .otherwise(F.lit(None))
    )

df_gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_OUTPUT_SUMMARY_TABLE)

print(f"Gold summary written: {df_gold_summary.count():,} rows")
display(df_gold_summary.orderBy("date", "cp_name", "shift").limit(20))


%md
### 5.2 Gold: Density Trend (for dashboard consumption)


### 5.3 Gold: WIM vs Conveyor Ratio (for anomaly detection)

In [0]:
# Write WIM vs Conveyor ratio to Gold table
df_wim_cv_ratio.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_WIM_CV_RATIO_TABLE)

print(f"WIM/CV ratio written: {df_wim_cv_ratio.count():,} rows -> {GOLD_WIM_CV_RATIO_TABLE}")
display(df_wim_cv_ratio.orderBy("date"))

In [0]:
df_density_gold = df_tonnage_with_delta \
    .filter(
        F.col("is_cv") &
        F.col("tonnage_delta").isNotNull() &
        F.col("delta_volume").isNotNull() &
        (F.col("delta_volume") > 0)
    ) \
    .groupBy("adjusted_date", "location", "flow_mode") \
    .agg(
        F.round(F.sum("tonnage_delta"), 2).alias("total_tonnage"),
        F.round(F.sum("delta_volume"), 2).alias("total_volume"),
    ) \
    .withColumn("density_realisasi",
        F.when(F.col("total_volume") > 0,
               F.round(F.col("total_tonnage") / F.col("total_volume"), 4))
         .otherwise(F.lit(None))
    ) \
    .withColumn("density_referensi", F.lit(DENSITY_REFERENCE)) \
    .withColumn("density_deviation_pct",
        F.round(
            (F.col("density_realisasi") - F.col("density_referensi")) / F.col("density_referensi") * 100,
            2
        )
    )

df_density_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_DENSITY_TREND_TABLE)

print(f"Density trend written: {df_density_gold.count():,} rows")
display(df_density_gold.orderBy("adjusted_date", "location").limit(20))


%md
---
### 5.3 Data Lineage & Statistics Summary


In [0]:
print("\n" + "=" * 70)
print("UC06 -- DATA LINEAGE SUMMARY")
print("=" * 70)
print(f"""
{'Zone':<10} {'Table':<45} {'Records':>12} {'Status':>10}
{'-'*80}
{'BRONZE':<10} {SRC_TONNAGE_LOGS:<45} {df_tonnage_logs_raw.count():>12,} {'SOURCE':>10}
{'BRONZE':<10} {SRC_TONNAGE_SNAPSHOT:<45} {df_tonnage_snapshot_raw.count():>12,} {'SOURCE':>10}
{'BRONZE':<10} {SRC_CP_ITEMS_LOGS:<45} {df_cp_items_raw.count():>12,} {'SOURCE':>10}
{'BRONZE':<10} {SRC_QUEUE_ASSIGNMENTS[:45]:<45} {df_queue_raw.count():>12,} {'SOURCE':>10}
{'BRONZE':<10} {SRC_WIM_TRANSACTIONS:<45} {df_wim_raw.count():>12,} {'SOURCE':>10}
{'-'*80}
{'SILVER':<10} {SILVER_TONNAGE_TABLE:<45} {df_tonnage_with_delta.count():>12,} {'SAVED':>10}
{'SILVER':<10} {SILVER_CP_STATUS_TABLE:<45} {df_cp_status_clean.count():>12,} {'SAVED':>10}
{'SILVER':<10} {SILVER_OUTPUT_PER_SHIFT_TABLE:<45} {df_output_per_shift.count():>12,} {'SAVED':>10}
{'-'*80}
{'GOLD':<10} {GOLD_OUTPUT_SUMMARY_TABLE:<45} {df_gold_summary.count():>12,} {'SAVED':>10}
{'GOLD':<10} {GOLD_DENSITY_TREND_TABLE:<45} {df_density_gold.count():>12,} {'SAVED':>10}
{'GOLD':<10} {GOLD_WIM_CV_RATIO_TABLE:<45} {df_wim_cv_ratio.count():>12,} {'SAVED':>10}
{'-'*80}
""")

print("\n== KEY METRICS (UC-06) ==")
print("1. Visualisasi Output Crushing Plant per shift (Direct vs Non-Direct)")
print(f"   -> {SILVER_OUTPUT_PER_SHIFT_TABLE}, {GOLD_OUTPUT_SUMMARY_TABLE}")
print("2. Pie Chart Proporsi Direct vs Non-Direct")
print(f"   -> {SILVER_TONNAGE_TABLE} (flow_mode column)")
print("3. Konsistensi Reading: WIM vs Conveyor Ratio")
print(f"   -> {GOLD_WIM_CV_RATIO_TABLE}")
print("4. Trend Density Realisasi vs Referensi (0.88)")
print(f"   -> {GOLD_DENSITY_TREND_TABLE}")
print()
print("Running hours & utilization:")
print(f"   -> {SILVER_CP_STATUS_TABLE}")

print("\nNotebook execution complete.")


%md
---
## 6. APPENDIX: Mapping Tables


%md
### A. CP ID -> CP Name Mapping (from items_logs)

| cp_id | CP Name | Equipment Count |
|-------|---------|-----------------|
| 1     | CP1     | Chain Feeder + Feeder Breaker |
| 2     | CP2     | Chain Feeder + Feeder Breaker |
| 3     | CP3     | Chain Feeder + Feeder Breaker |
| 4     | CP4     | Chain Feeder + Feeder Breaker |
| 5     | CP5     | Chain Feeder + Feeder Breaker |
| 6     | CP6     | Primary Crusher + CF100 |
| 7     | CP7     | CF108 + OMS PC01 |
| 8     | CP8     | Chain Feeder + Feeder Breaker |
| 9     | CP9     | Chain Feeder + Feeder Breaker |
| 11    | CP2New  | Conveyor Status x2 |

### B. Location Classification (from tonage_logs)

| Type          | Locations |
|---------------|-----------|
| Crushing Plant | CP1, CP6, CP7, CP8, CP9 |
| Direct Conveyor | CV14, CV22, CV12, CV105, CV106, CV107, CV116 |
| Non-Dir Conveyor | CV15A, CV15B, CV23A, CV23B |

### C. Flow Mode Classification

| flow_mode   | Description |
|-------------|-------------|
| cp_output   | Tonnage from CP totalizer (CP production) |
| direct      | Tonnage from conveyors directly to shipment |
| non_direct  | Tonnage from stockpile reclaim conveyors |

### D. Shift Definition

| Shift  | Time                  | Notes |
|--------|-----------------------|-------|
| Pagi   | 06:00 AM - 05:59 PM   | Morning  |
| Malam  | 06:00 PM - 05:59 AM   | Night (00:00-05:59 portion -> assigned to previous date) |

### E. Key Calculations

| Metric           | Formula |
|------------------|---------|
| Delta Volume     | totalizer_today[t] - totalizer_today[t-1] |
| Tonnage          | Volume x 0.88 (Density BIB) |
| Throughput (tph) | Tonnage_delta / (time_delta / 3600) |
| Running Hours    | (running_readings x 30s) / 3600 |
| WIM/CV Ratio     | SUM(WIM_truck_tonnage) / SUM(conveyor_tonnage) |
| Density Realisasi| Tonnage / Volume (should approximate 0.88) |
